In this notebook, we evaluate the performance of our ensemble for Table 2 on the holdout set. We compute the bootstrapped RMSE and MAE on 10,000 samples with its 95% confidence interval on the holdout set.

In [5]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import rcParams
import numpy as np
import pandas as pd
import os

In [6]:
def compute_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def compute_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def bootstrapped_metric(y_true, y_pred, metric_fn, n_iterations=10000, ci=95, seed=None):
    rng = np.random.default_rng(seed)
    n_size = len(y_true)
    boot_stats = np.empty(n_iterations)

    for i in range(n_iterations):
        indices = rng.integers(0, n_size, size=n_size)
        boot_stats[i] = metric_fn(y_true[indices], y_pred[indices])

    lower = np.percentile(boot_stats, (100 - ci) / 2)
    upper = np.percentile(boot_stats, 100 - (100 - ci) / 2)
    return boot_stats.mean(), (lower, upper)

def bootstrapped_rmse(y_true, y_pred, n_iterations=10000, ci=95, seed=42):
    return bootstrapped_metric(y_true, y_pred, compute_rmse, n_iterations, ci, seed)

def bootstrapped_mae(y_true, y_pred, n_iterations=10000, ci=95, seed=42):
    return bootstrapped_metric(y_true, y_pred, compute_mae, n_iterations, ci, seed)

In [7]:
path = "/home/jusdai/cil-project/results/benchmark/full_pipeline.npz"

In [8]:
data = np.load(path)
preds = data["holdout_pred"]
targets = data["y_holdout"]

mean_rmse, rmse_ci = bootstrapped_rmse(targets, preds, seed=0)
mean_mae,  mae_ci  = bootstrapped_mae(targets, preds, seed=0)

df_results = pd.DataFrame([{
    "RMSE": mean_rmse,
    "RMSE_Lower": rmse_ci[0],
    "RMSE_Upper": rmse_ci[1],
    "RMSE_Error": (rmse_ci[1] - rmse_ci[0]) / 2,
    "MAE": mean_mae,
    "MAE_Lower": mae_ci[0],
    "MAE_Upper": mae_ci[1],
    "MAE_Error": (mae_ci[1] - mae_ci[0]) / 2,
}])

In [9]:
df_results

,RMSE,RMSE_Lower,RMSE_Upper,RMSE_Error,MAE,MAE_Lower,MAE_Upper,MAE_Error
0,0.874405,0.871597,0.877167,0.002785,0.685217,0.683019,0.687392,0.002187
